[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/03-OCRfacturas.ipynb)


📁 [Recursos del módulo (Google Drive)](https://drive.google.com/drive/folders/1t4VQB_6FL5wpJTPIw6dprTnCPt0uDjYg?usp=drive_link)


# 03 — Parsing y Extraction: comparando métodos de OCR

En **`02-PDF_reporte`** vimos que **parsing** (`pypdf`) solo sirve si el PDF ya trae texto seleccionable. En cuanto el "documento" es en realidad una foto, `extract_text()` regresa (casi) nada — la información sigue ahí, pero como píxeles, no como caracteres.

Antes de meternos a OCR, el primer paso **siempre** es preguntar: *¿este PDF ya trae texto seleccionable?* Si sí, `pypdf` te resuelve todo gratis e instantáneo — no necesitas nada de lo que sigue. Empezamos probando eso, y solo si falla pasamos a las 4 rondas de OCR/Extraction.

| Ronda | Método | Qué hace |
|---|---|---|
| 1 | **Tesseract** | Parsing: OCR "clásico", sin deep learning. |
| 2 | **EasyOCR** | Parsing: OCR con redes neuronales. |
| 3 | **[Falcon-OCR](https://huggingface.co/tiiuae/Falcon-OCR)** | Extraction: modelo de Hugging Face especializado en documentos — corre local, pero necesita GPU. |
| 4 | **[LlamaExtract](https://cloud.llamaindex.ai)** | Extraction: servicio en la nube, le das el esquema de campos que quieres y te regresa JSON. Sin GPU, pero necesitas tu propia cuenta (gratis). |

Comparamos **a simple vista**: corres cada ronda, ves lo que imprime, y lo comparas contra la factura real (ábrela junto al notebook).


## Antes de empezar: activa GPU

**En Colab, activa GPU desde ahora** — `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`. Sí vamos a usar la GPU en este notebook (EasyOCR y Falcon-OCR corren con ella). Si esperas hasta la Ronda 3 (Falcon-OCR) para activarla, Colab **reinicia el entorno de ejecución** y pierdes todo lo que ya corriste; tendrías que empezar de nuevo desde arriba.


## Archivos que necesitas

1. `factura_1_seleccionable.pdf`
2. `factura_1_no_seleccionable.pdf`
3. `factura_2.pdf`
4. `factura_3.pdf`
5. `ey_100_casos_rentables_ia_2026.pdf`


## Paso 0 — ¿el texto ya es seleccionable?

Muchas facturas reales (las que arma un sistema de facturación, como las tuyas) **ya traen texto seleccionable** de fábrica — no son una foto ni un escaneo. En ese caso no hace falta nada de OCR: `pypdf` (parsing, lo que vimos en `02`) resuelve todo solo.

Probemos con `factura_1_seleccionable.pdf` (misma factura que usamos en todo el notebook, pero con texto real dentro del PDF, no una imagen):


In [ ]:
%pip install -q pypdf

from pypdf import PdfReader

lector = PdfReader("factura_1_seleccionable.pdf")
texto = lector.pages[0].extract_text()
print(texto)


**Funcionó, y no usamos ni OCR ni GPU ni ninguna cuenta externa.** Si tu factura ya viene así, aquí termina el trabajo — con esto y una regex (como la que vimos en `02`) ya tienes tus campos.

Pero **no todas las facturas vienen así**: una que se escaneó, o que alguien fotografió con el celular, es solo una imagen — aunque a simple vista se vea igual, no hay texto que copiar. Ahí es donde `pypdf` deja de servir y entran las 4 rondas de este notebook. Probemos con la **misma factura**, pero guardada solo como imagen (`factura_1_no_seleccionable.pdf`):


In [ ]:
lector_factura_1 = PdfReader("factura_1_no_seleccionable.pdf")
texto_factura_1 = lector_factura_1.pages[0].extract_text()
print(f"Caracteres extraídos: {len(texto_factura_1)}")


**Cero caracteres.** Mismo código, mismo `pypdf`, misma factura -- la diferencia está en cómo se guardó el documento, no en la herramienta. De aquí en adelante trabajamos con `factura_1_no_seleccionable.pdf`, `factura_2.pdf` y `factura_3.pdf`.


In [ ]:
%pip install -q pypdfium2

import time

import pypdfium2 as pdfium
from PIL import Image

FACTURAS = ["factura_1_no_seleccionable.pdf", "factura_2.pdf", "factura_3.pdf"]


def pdf_a_imagen(ruta_pdf):
    return pdfium.PdfDocument(ruta_pdf)[0].render(scale=2).to_pil().convert("RGB")


## Lo que dice cada factura (el "dato real")

Ábrelas junto al notebook (`factura_1_no_seleccionable.pdf`, etc.) y compáralas contra lo que imprima cada ronda:

| Factura | Folio | Fecha | Emisor | Total |
|---|---|---|---|---|
| `factura_1_no_seleccionable.pdf` | 001 | 2026-03-14 | Laura Ximena Reyes Cortés | $ 15,660.00 |
| `factura_2.pdf` | 002 | 2026-05-02 | Soluciones Digitales del Bajío | $ 9,512.00 |
| `factura_3.pdf` | 003 | 2026-07-21 | Miguel Ángel Torres Domínguez | $ 25,520.00 |


## Ronda 1 — Tesseract

**Ventajas:** gratis, instantáneo, no necesita GPU.

**Desventajas:** solo texto plano (tú tienes que encontrar el dato); le cuesta con fotos de mala calidad.


In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
%pip install -q pytesseract


In [ ]:
import pytesseract

for nombre in FACTURAS:
    t0 = time.time()
    texto = pytesseract.image_to_string(pdf_a_imagen(nombre))
    print(f"--- {nombre} ({time.time() - t0:.1f}s) ---")
    print(texto)
    print()


## De parsing a extraction: un ejemplo

El texto de arriba (parsing) es correcto, pero es un bloque de texto -- tú tienes que encontrar el total ahí adentro. Extraer significa ir un paso más allá: pasar de "aquí está todo el texto" a "el total es $15,660.00". Una línea de regex sobre el texto de factura_1_no_seleccionable.pdf alcanza para hacerlo a mano:


In [ ]:
import re

texto_factura_1 = pytesseract.image_to_string(pdf_a_imagen("factura_1_no_seleccionable.pdf"))
coincidencias = re.findall(r"[TtOo]otal:?\s*\$?\s*([\d.,]+)", texto_factura_1)
print("Total encontrado:", coincidencias[-1] if coincidencias else "no encontrado")


(Buscamos "total" y tomamos la última coincidencia porque "Subtotal" también contiene la palabra "total".) Esto ya da una idea de por qué conviene un método que regrese el dato directo, sin regex -- eso es lo que hace la ronda 4 (LlamaExtract).


## Ronda 2 — EasyOCR

**Ventajas:** mejor que Tesseract con fotos "reales" (ángulos, fondos).

**Desventajas:** más lento, descarga un modelo de ~500 MB la primera vez.


In [ ]:
%pip install -q easyocr


In [ ]:
import numpy as np
import easyocr

lector_easyocr = easyocr.Reader(["es"], gpu=True)

for nombre in FACTURAS:
    t0 = time.time()
    lineas = lector_easyocr.readtext(np.array(pdf_a_imagen(nombre)), detail=0)
    print(f"--- {nombre} ({time.time() - t0:.1f}s) ---")
    print("\n".join(lineas))
    print()


### Ver las cajas: qué detectó EasyOCR

EasyOCR primero detecta regiones de texto (cajas) y luego reconoce qué dice cada una. Dibujemos esas cajas sobre la factura, en su propia celda -- para no mezclar la imagen con el texto que imprimimos arriba.


In [ ]:
from PIL import ImageDraw

imagen_con_cajas = pdf_a_imagen("factura_1_no_seleccionable.pdf").convert("RGB")
dibujo = ImageDraw.Draw(imagen_con_cajas)

detecciones = lector_easyocr.readtext(np.array(imagen_con_cajas), detail=1)
for caja, texto, confianza in detecciones:
    dibujo.polygon([tuple(p) for p in caja], outline="red", width=3)

imagen_con_cajas


## Ronda 3 -- Falcon-OCR, un modelo de Hugging Face

Falcon-OCR (de TII) es un modelo chico (300M parametros) especializado en documentos: en vez de texto plano, le pides directo texto, fórmula (LaTeX) o tabla (HTML).

**Necesita GPU** (ya la activaste al inicio del notebook, ¿verdad?).

**Ventajas:** entiende tablas sin regex; modelo chico.

**Desventajas:** necesita GPU; lento incluso en el T4 gratis de Colab (por eso aquí solo probamos **una** factura, no las 3); solo tiene esos 3 modos fijos, no es conversacional.

Este es el código tal cual viene en la ficha del modelo en Hugging Face (https://huggingface.co/tiiuae/Falcon-OCR):


In [ ]:
%pip install -q -U transformers accelerate


In [ ]:
import warnings

import torch
from transformers import AutoModelForCausalLM

warnings.filterwarnings("ignore")

modelo_falcon = AutoModelForCausalLM.from_pretrained(
    "tiiuae/Falcon-OCR",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

t0 = time.time()
texto = modelo_falcon.generate(pdf_a_imagen("factura_1_no_seleccionable.pdf"))[0]
print(f"({time.time() - t0:.1f}s)")
print(texto)


### El truco: pedirle la tabla directo

Con category="table" te regresa la tabla de renglones ya en HTML. Probemos con factura_3.pdf (la más difícil de las 3 -- mala composición a propósito):


In [ ]:
from IPython.display import HTML, display

tabla_html = modelo_falcon.generate(pdf_a_imagen("factura_3.pdf"), category="table")[0]
display(HTML(tabla_html))


## Ronda 4 -- LlamaExtract (servicio en la nube, sin GPU)

LlamaExtract (de la startup LlamaIndex) es distinto a todo lo anterior: no corre en tu computadora ni en Colab, corre en la nube. Le defines los campos que quieres (un esquema) y te regresa JSON.

Necesitas una cuenta gratis en https://cloud.llamaindex.ai (plan gratis: 10,000 usos al mes) y tu propia API key. La celda de abajo trae `api_key = ""` vacío -- pega ahí tu key.

**Ventajas:** no necesita GPU; JSON ya tipado, sin regex.

**Desventajas:** depende de internet y de una cuenta externa; cada quien necesita su propia API key.


In [ ]:
%pip install -q "llama-cloud>=2.8"


In [ ]:
from pydantic import BaseModel
from llama_cloud import LlamaCloud

api_key = ""  # pega aqui tu API key de LlamaCloud
cliente_llama = LlamaCloud(api_key=api_key)


class Factura(BaseModel):
    folio: str
    fecha: str
    emisor: str
    total: float


In [ ]:
for nombre in FACTURAS:
    archivo = cliente_llama.files.create(file=nombre, purpose="extract")
    trabajo = cliente_llama.extract.create(
        file_input=archivo.id,
        configuration={"data_schema": Factura.model_json_schema(), "extraction_target": "per_doc"},
    )
    while trabajo.status not in ("COMPLETED", "FAILED", "CANCELLED"):
        time.sleep(2)
        trabajo = cliente_llama.extract.get(trabajo.id)
    print(f"--- {nombre} ---")
    print(trabajo.extract_result)
    print()


### Importa como capturaste el documento: PDF limpio vs. foto del PDF

En la vida real, muchas veces el punto de partida es un PDF y alguien lo imprime y le toma una foto con el celular en vez de mandar el archivo digital. Probamos Falcon-OCR sobre la misma página, capturada de dos formas.


In [ ]:
import numpy as np
from io import BytesIO

pagina_limpia = pdf_a_imagen("ey_100_casos_rentables_ia_2026.pdf")

# Simulamos una foto de celular: rotada, mas chica, con sombra y compresion agresiva.
pagina_foto = pagina_limpia.rotate(6, expand=True, fillcolor=(255, 255, 255))
pagina_foto = pagina_foto.resize((pagina_foto.width // 2, pagina_foto.height // 2))
gradiente = np.tile(np.linspace(0.65, 1.0, pagina_foto.width), (pagina_foto.height, 1))
arreglo = np.array(pagina_foto).astype(float)
for canal in range(3):
    arreglo[:, :, canal] *= gradiente
buffer = BytesIO()
Image.fromarray(np.clip(arreglo, 0, 255).astype("uint8")).save(buffer, format="JPEG", quality=40)
buffer.seek(0)
pagina_foto = Image.open(buffer).convert("RGB")

print("--- PDF renderizado limpio ---")
print(modelo_falcon.generate(pagina_limpia)[0][:400])
print()
print("--- Foto simulada del PDF ---")
print(modelo_falcon.generate(pagina_foto)[0][:400])


Si la segunda transcripción tiene más errores o le faltan pedazos, esa es la lección: el modelo importa, pero la calidad de la captura del documento importa tanto o más.


## Cierre

Corriste las mismas 3 facturas por 4 métodos. Ahora compara tú mismo, a ojo (con la tabla de "lo que dice cada factura" de arriba, o abriendo los PDF directamente):

- ¿A cuál método le costó más el folio, la fecha o el total?
- ¿Cuál te dio el dato ya limpio (JSON, en la ronda 4) y a cuál le tuviste que sacar tú el dato del texto (rondas 1 a 3)?
- ¿Cuál tardó más? ¿Cuál necesitó GPU o internet y cuál no?

No hay una respuesta única de "cuál es el mejor" -- depende de si tienes GPU, si te preocupa mandar documentos a la nube, cuántas facturas vas a procesar, y qué tan complicado es el layout.

En producción, muchas empresas ya resuelven esto con **servicios administrados** (Google Document AI, AWS Textract, Azure Document Intelligence, o el mismo LlamaExtract) que ya vienen entrenados para facturas -- la ventaja de hacerlo "a mano" aquí es entender qué está pasando por dentro antes de delegarlo a una caja negra.


## ¿Y cuál me conviene entonces?

La respuesta honesta es: depende. No hay un método que gane en todo — cada uno cede algo a cambio de algo más. Estos son los cuatro criterios que más cambian la respuesta:

![Comparación cualitativa de los 4 métodos en costo, velocidad, precisión, privacidad y facilidad de instalación](https://raw.githubusercontent.com/milioe/casos-ia-ibero-diplomado/main/modulo_4/comparacion_metodos_ocr.png)

**Presupuesto.** Tesseract, EasyOCR y Falcon-OCR son gratis (aunque Falcon-OCR necesita una GPU, que si no la tienes localmente, cuesta rentarla). LlamaExtract tiene capa gratis, pero es un servicio: si tu volumen crece, empiezas a pagar por documento — y ahí es donde muchos servicios comerciales de OCR se vuelven caros rápido cuando pasas de decenas a miles o cientos de miles de documentos al mes. Si vas a procesar mucho volumen de forma constante, vale la pena hacer ese cálculo **antes** de comprometerte a un servicio por documento.

**Privacidad y confidencialidad.** Este es el criterio que más rápido descarta opciones. Si tus documentos traen información sensible — precios entre empresas relacionadas, fórmulas o procesos internos, datos de clientes con acuerdos de confidencialidad — mandarlos a un servicio en la nube significa que esa información sale de tu infraestructura.

- **Tesseract, EasyOCR y Falcon-OCR** corren **100% local** en este notebook: no es una promesa de nadie, es un hecho técnico — tu documento nunca sale de tu máquina, así que no hay "política de privacidad" que revisar porque no hay ningún servidor externo involucrado.
- **LlamaExtract sí es un servicio en la nube**, así que ahí sí aplica revisar la letra chica. No hay que quedarse con "seguro dicen que no entrenan" — esto es lo que dicen sus [Términos de Servicio](https://www.llamaindex.ai/legal/terms-of-service) (secciones 3.2 y 7.1), tal cual:
  - *"We do not train any models on User Content."*
  - *"LlamaIndex will only retain such User Content ... in S3 for up to 48 hours after processing."*
  - *"LlamaIndex will encrypt the User Content both in transit and at rest."*
  - Certificaciones verificables (SOC 2, GDPR, HIPAA) en su [Trust Center](https://security.llamaindex.ai/).

**¿Cómo revisas esto para cualquier otro servicio que usen?** Regla rápida: **app gratuita de consumidor** (ChatGPT free, etc.) → probablemente sí puede usar tus datos para entrenar, a menos que lo desactives en configuración. **API de pago / plan empresarial** → los proveedores grandes casi siempre declaran explícitamente que no entrenan con datos de la API por default — pero *revisa los Términos de Servicio del producto específico que vas a usar*, no la política general de la empresa, y busca el toggle de "Data controls" en el dashboard de tu cuenta. Para uso empresarial serio, pide un **DPA** (Data Processing Agreement) firmado — es lo legalmente vinculante, no una frase en un blog.

**Volumen.** Si procesas facturas ocasionalmente (unas cuantas a la semana), casi cualquier método sirve — hasta revisarlas a mano puede ser razonable. Si necesitas procesar miles de documentos al día de forma automática (por ejemplo, en logística, comercio exterior o cualquier operación con mucho papeleo repetitivo), entonces la velocidad y el costo marginal por documento empiezan a pesar mucho más que la comodidad de instalación.

**Hardware disponible.** Sin GPU, Falcon-OCR deja de ser una opción razonable (correría muy lento en CPU). Sin internet confiable o sin poder crear cuentas externas, LlamaExtract tampoco. Tesseract y EasyOCR son los que menos piden: CPU y ya.

**Entonces, ¿por dónde empezar?** Si nunca has hecho esto: arranca con Tesseract o EasyOCR — son gratis, locales, y te dejan ver rápido si el problema es viable. Si el layout es complicado (tablas, tickets, tamaños variados) y tienes GPU disponible, Falcon-OCR te ahorra mucho trabajo de regex. Si la privacidad no es negociable, descarta cualquier servicio en la nube sin importar qué tan bien evalúe en los demás criterios. Y si vas a procesar mucho volumen, haz la cuenta del costo por documento **antes** de escalar, no después.
